In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta

1. Generate Simulated Base Traffic Data (Junction Volumes)

In [2]:
np.random.seed(42)
date_range = pd.date_range(start="2026-06-01", end="2026-06-07 23:00:00", freq="H")
junctions = ['Junction_A', 'Junction_B', 'Junction_C']

C:\Users\patil\AppData\Local\Temp\ipykernel_23436\260729296.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_range = pd.date_range(start="2026-06-01", end="2026-06-07 23:00:00", freq="H")


In [3]:
traffic_records = []
for dt in date_range:
    for junc in junctions:
        base_vol = 200 + 300 * np.sin(2 * np.pi * dt.hour / 24) + np.random.randint(-50, 50)
        traffic_records.append({
            "Timestamp": dt,
            "Junction": junc,
            "Traffic_Volume": max(10, int(base_vol))
        })

df_traffic = pd.DataFrame(traffic_records)

2. Generate and Integrate Weather Data

In [4]:
weather_records = []
for dt in date_range:
    temp = 22 + 8 * np.sin(2 * np.pi * dt.hour / 24) + np.random.uniform(-2, 2)
    precipitation = 0.0 if np.random.rand() > 0.15 else np.random.uniform(0.5, 5.0) # 15% chance of rain
    humidity = 60 + 20 * np.cos(2 * np.pi * dt.hour / 24) + np.random.uniform(-5, 5)
    wind_speed = np.random.uniform(5, 25)

    weather_records.append({
        "Timestamp": dt,
        "Temperature_C": round(temp, 1),
        "Precipitation_mm": round(precipitation, 2),
        "Humidity_Pct": round(min(100, max(0, humidity)), 1),
        "Wind_Speed_kmh": round(wind_speed, 1)
    })
df_weather = pd.DataFrame(weather_records)

3. Generate and Integrate Event Data

In [5]:
event_records = []
for dt in date_range:
    is_event = 0
    event_type = "None"
    
    if dt.date() == datetime.strptime("2026-06-03", "%Y-%m-%d").date() and 18 <= dt.hour <= 22:
        is_event = 1
        event_type = "Sports_Event"
    elif dt.date() == datetime.strptime("2026-06-05", "%Y-%m-%d").date():
        is_event = 1
        event_type = "Public_Holiday"
        
    event_records.append({
        "Timestamp": dt,
        "Is_Event": is_event,
        "Event_Type": event_type
    })
df_events = pd.DataFrame(event_records)

4. Pipeline Merging (Synchronization based on Timestamps)

In [6]:
df_integrated = pd.merge(df_traffic, df_weather, on="Timestamp", how="left")
df_integrated = pd.merge(df_integrated, df_events, on="Timestamp", how="left")

5. Handle Data Quality (Introduce and Clean Missing Values / Outliers)

In [7]:
df_integrated.loc[df_integrated.sample(frac=0.02).index, 'Traffic_Volume'] = np.nan

In [8]:
df_integrated['Traffic_Volume'] = df_integrated.groupby('Junction')['Traffic_Volume'].transform(lambda x: x.fillna(x.median()))

6. Normalization / Scaling numerical columns

In [10]:
scaler = MinMaxScaler()
numerical_cols = ['Traffic_Volume', 'Temperature_C', 'Precipitation_mm', 'Humidity_Pct', 'Wind_Speed_kmh']
scaled_features = scaler.fit_transform(df_integrated[numerical_cols])

In [11]:
for i, col in enumerate(numerical_cols):
    df_integrated[f'{col}_Scaled'] = scaled_features[:, i]

In [14]:
output_filename = "Integrated_Dataset_Harshita.csv"
df_integrated.to_csv(output_filename, index=False)
print(f"Success! {output_filename} has been generated with {len(df_integrated)} rows.")

Success! Integrated_Dataset_Harshita.csv has been generated with 504 rows.
